[Guide Index](README.md) | [OWL 2 Manchester Syntax guide](02.08-manchester-syntax.ipynb)

# Appendix C: OWL 2 Manchester Syntax reference

This appendix goes through the OWL 2 Manchester Syntax notation construct by construct. Each section says what a construct means, shows a small piece of Manchester Syntax text, and shows the RDF that StarLayer produces from it. The [Manchester Syntax guide](02.08-manchester-syntax.ipynb) covers the two directions, text to RDF and RDF back to text, and this appendix is the reference to look constructs up in.

An ontology in Manchester Syntax is a list of frames. Each frame names one entity (a class, a property or an individual) and lists clauses that say things about it. Each clause is an *axiom*: a statement the ontology asserts to be true. A reasoner uses axioms to draw conclusions, for example that an individual belongs to a class; the conclusions appear only after inference (see [02.06](02.06-graphs-inferencing.ipynb) and [02.07](02.07-owl-dl-reasoning.ipynb)). Axioms do not reject data that breaks them. OWL also does not assume that what is not stated is false (the open world assumption): a car with no stated owner is not an error, it has an owner that is not known. To reject data that breaks a rule, use SHACL (see [05.01](05.01-validating-with-shacl-1.2.ipynb)).

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). No extra runtime dependencies are needed. Run cells from top to bottom.

A `StarLayerGraph` starts with no prefix bindings, so the setup cell sets default prefixes (`:`, `owl:`, `rdfs:`) that every new graph starts with. The `show()` helper parses a Manchester Syntax snippet and prints the RDF it maps to as Turtle. Each example is a separate snippet, so it declares the entities it uses.

In [ ]:
from starlayer.graph import Namespace, OWL, RDFS, StarLayerGraph, set_default_prefixes

EX = Namespace("http://example.org/vehicles#")

# every graph made after this call starts with these prefixes bound
set_default_prefixes({"": EX, "owl": OWL, "rdfs": RDFS})


def show(manchester_text):
    """Parse a Manchester Syntax snippet and print the RDF it maps to."""
    g = StarLayerGraph()
    g.parse(data=f"Prefix: : <{EX}>\n" + manchester_text, format="manchester")
    print(g.serialize(format="turtle", pretty=True))
    return g

## 1. Classes

A class is a set of individuals. Each `Class:` frame declares one class (`owl:Class`) and its clauses add axioms about it. An axiom is a statement the ontology asserts to be true. The parser writes only the triples the text states. What follows from them, such as an individual of `Car` also being a `Vehicle`, appears when a reasoner draws that conclusion (see [02.06](02.06-graphs-inferencing.ipynb) and [02.07](02.07-owl-dl-reasoning.ipynb)).

### 1.1 `SubClassOf:` and `SuperClassOf:`

`SubClassOf: Vehicle` means every `Car` is a `Vehicle`. `SuperClassOf:` is the same statement spelled in the other direction: `Vehicle SuperClassOf: Motorcycle` means the same as `Motorcycle SubClassOf: Vehicle`.

Below, two classes are declared as subclasses of `Vehicle`, one with `SubClassOf:` and one with `SuperClassOf:`. Both produce the same kind of triple, `rdfs:subClassOf`.

In [ ]:
_ = show("""
Class: Vehicle
    SuperClassOf: Motorcycle
Class: Car
    SubClassOf: Vehicle
Class: Motorcycle
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class ;
    rdfs:subClassOf :Vehicle .

:Motorcycle a owl:Class ;
    rdfs:subClassOf :Vehicle .

:Vehicle a owl:Class .



### 1.2 `DisjointWith:`

`DisjointWith: Car` means nothing can be both a `Truck` and a `Car`. If an individual is stated to be both, the ontology is inconsistent and a DL reasoner reports it.

Below, `Truck` is a `Vehicle` that is disjoint with `Car`.

In [ ]:
_ = show("""
Class: Vehicle
Class: Car
    SubClassOf: Vehicle
Class: Truck
    SubClassOf: Vehicle
    DisjointWith: Car
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class ;
    rdfs:subClassOf :Vehicle .

:Truck a owl:Class ;
    rdfs:subClassOf :Vehicle ;
    owl:disjointWith :Car .

:Vehicle a owl:Class .



## 2. Properties

A property relates individuals. An object property relates an individual to another individual, and a data property relates an individual to a literal value. Properties have their own frames, `ObjectProperty:` and `DataProperty:`.

### 2.1 `Domain:`, `Range:` and `Characteristics:`

`Domain: Vehicle` means anything that has the property is concluded to be a `Vehicle`. It is not a check: using `hasEngine` on something makes it a `Vehicle`. `Range: Engine` means the values of the property are concluded to be `Engine`s. `Characteristics:` states logical properties of the property. `Functional` means each individual has at most one value for it. The others are `InverseFunctional`, `Transitive`, `Symmetric`, `Asymmetric`, `Reflexive` and `Irreflexive`.

Below, an object property with a domain, a range and a characteristic.

In [ ]:
_ = show("""
Class: Vehicle
Class: Engine
ObjectProperty: hasEngine
    Domain: Vehicle
    Range: Engine
    Characteristics: Functional
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Engine a owl:Class .

:Vehicle a owl:Class .

:hasEngine a owl:FunctionalProperty, owl:ObjectProperty ;
    rdfs:domain :Vehicle ;
    rdfs:range :Engine .



### 2.2 `InverseOf:`

`InverseOf: hasEngine` means `poweredBy` runs the other way: if A `hasEngine` B, then B `poweredBy` A.

Below, `poweredBy` is declared as the inverse of `hasEngine`.

In [ ]:
_ = show("""
ObjectProperty: hasEngine
ObjectProperty: poweredBy
    InverseOf: hasEngine
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:hasEngine a owl:ObjectProperty .

:poweredBy a owl:ObjectProperty ;
    owl:inverseOf :hasEngine .



### 2.3 `SubPropertyOf:` and `SuperPropertyOf:`

`SubPropertyOf: hasEngine` means whenever two individuals are linked by `hasSubEngine`, they are also linked by `hasEngine`. `SuperPropertyOf:` is the same statement in the other direction.

Below, the same relationship is written in both directions.

In [ ]:
_ = show("""
ObjectProperty: hasEngine
ObjectProperty: hasSubEngine
    SubPropertyOf: hasEngine
ObjectProperty: hasComponent
    SuperPropertyOf: hasSubEngine
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:hasComponent a owl:ObjectProperty .

:hasEngine a owl:ObjectProperty .

:hasSubEngine a owl:ObjectProperty ;
    rdfs:subPropertyOf :hasComponent, :hasEngine .



### 2.4 `SubPropertyChain:`

`SubPropertyChain: hasParent o hasParent` means that if A `hasParent` B and B `hasParent` C, then A `hasGrandparent` C. It applies to object properties only.

Below, `hasGrandparent` is the chain of two `hasParent` links.

In [ ]:
_ = show("""
ObjectProperty: hasParent
ObjectProperty: hasGrandparent
    SubPropertyChain: hasParent o hasParent
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:hasGrandparent a owl:ObjectProperty ;
    owl:propertyChainAxiom ( :hasParent :hasParent ) .

:hasParent a owl:ObjectProperty .



### 2.5 Data properties

A data property takes the same clauses, but its range is a datatype, such as `xsd:decimal`. Datatypes and data ranges are described in section 6.

Below, a data property with a domain, a datatype range and a characteristic.

In [ ]:
_ = show("""
Class: Vehicle
DataProperty: hasWeightKg
    Domain: Vehicle
    Range: xsd:decimal
    Characteristics: Functional
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:Vehicle a owl:Class .

:hasWeightKg a owl:DatatypeProperty, owl:FunctionalProperty ;
    rdfs:domain :Vehicle ;
    rdfs:range xsd:decimal .



## 3. Individuals

An individual is a particular thing. An `Individual:` frame says what it is and what is known about it.

### 3.1 `Types:` and `Facts:`

`Types: Car` means the individual is a member of the class `Car`. `Facts:` states facts about the individual: a link to another individual (`hasOwner Alice`), or a data value (`hasVIN "..."`).

Below, an individual with a type and two facts.

In [ ]:
_ = show("""
Class: Car
ObjectProperty: hasOwner
DataProperty: hasVIN
Individual: Alice

Individual: mycar
    Types: Car
    Facts: hasOwner Alice, hasVIN "1HGCM82633A004352"
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Alice a owl:NamedIndividual .

:Car a owl:Class .

:hasOwner a owl:ObjectProperty .

:hasVIN a owl:DatatypeProperty .

:mycar a :Car, owl:NamedIndividual ;
    :hasOwner :Alice ;
    :hasVIN "1HGCM82633A004352" .



### 3.2 Negative facts

A leading `not` states that a fact is false. It maps to an `owl:NegativePropertyAssertion` and is not silently dropped. `not hasOwner Bob` says Bob is not the owner.

Below, the same individual with one negative fact.

In [ ]:
_ = show("""
Class: Car
ObjectProperty: hasOwner
Individual: Bob

Individual: mycar
    Types: Car
    Facts: not hasOwner Bob
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Bob a owl:NamedIndividual .

:Car a owl:Class .

:hasOwner a owl:ObjectProperty .

:mycar a :Car, owl:NamedIndividual .

_:b0 a owl:NegativePropertyAssertion ;
    owl:assertionProperty :hasOwner ;
    owl:sourceIndividual :mycar ;
    owl:targetIndividual :Bob .



### 3.3 `SameAs:` and `DifferentFrom:`

`SameAs: mycar2` means the two names refer to the same individual. `DifferentFrom: Bob` means the two are different individuals. OWL does not assume that different names mean different individuals (no unique name assumption). Two names are only known to be different when the ontology says so.

Below, one individual is the same as another and different from a third.

In [ ]:
_ = show("""
Individual: Bob
Individual: mycar2

Individual: mycar
    SameAs: mycar2
    DifferentFrom: Bob
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Bob a owl:NamedIndividual .

:mycar a owl:NamedIndividual ;
    owl:differentFrom :Bob ;
    owl:sameAs :mycar2 .

:mycar2 a owl:NamedIndividual .



## 4. Keys and disjoint unions

Two more axioms about a class.

### 4.1 `HasKey:`

`HasKey: hasVIN` means a property (or a set of properties) identifies the named individuals of the class: two named individuals of the class with the same value for every key property are concluded to be the same individual. It is a conclusion a reasoner draws, not a uniqueness check on the data.

Below, a class with a key.

In [ ]:
_ = show("""
DataProperty: hasVIN

Class: Vehicle
    HasKey: hasVIN
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Vehicle a owl:Class ;
    owl:hasKey ( :hasVIN ) .

:hasVIN a owl:DatatypeProperty .



### 4.2 `DisjointUnionOf:`

`DisjointUnionOf: Car, Truck, Motorcycle` means a `Vehicle` is exactly a `Car`, a `Truck` or a `Motorcycle`, and nothing is in more than one of them. The subclasses are disjoint and together make up the class.

Below, `Vehicle` is the disjoint union of three classes.

In [ ]:
_ = show("""
Class: Car
Class: Truck
Class: Motorcycle

Class: Vehicle
    DisjointUnionOf: Car, Truck, Motorcycle
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class .

:Motorcycle a owl:Class .

:Truck a owl:Class .

:Vehicle a owl:Class ;
    owl:disjointUnionOf ( :Car :Truck :Motorcycle ) .



## 5. Class expressions

An expression describes a class by its conditions: the set of all individuals that meet them. It can be used wherever a class name can, for example after `SubClassOf:` or `EquivalentTo:`.

### 5.1 `and`, `or` and `not`

`Car and Truck` is the individuals in both classes (the intersection). `Car or Truck` is the individuals in either class (the union). `not Truck` is every individual that is not a `Truck` (the complement).

Below, `ElectricCar` is a `Car` that is not a `Truck`.

In [ ]:
_ = show("""
Class: Car
Class: Truck

Class: ElectricCar
    SubClassOf: Car and (not Truck)
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class .

:ElectricCar a owl:Class ;
    rdfs:subClassOf [
        a owl:Class ;
        owl:intersectionOf (
            :Car
            [
                a owl:Class ;
                owl:complementOf :Truck
            ]
        )
    ] .

:Truck a owl:Class .



### 5.2 `EquivalentTo:` and `that`

`EquivalentTo:` is a definition in both directions: an individual is a `FamilyVehicle` if and only if it is a `Car` or a `Truck`. `SubClassOf:` works in one direction only, so an `EquivalentTo:` definition lets a reasoner place individuals in the class as well as draw conclusions from membership. `that` is a synonym for `and`, used to refine a description in a sentence-like way ("a family vehicle that has airbags").

Below, `FamilyVehicle` is defined as a `Car` or a `Truck`, and `SafeFamilyVehicle` as a `FamilyVehicle` that has airbags.

In [ ]:
_ = show("""
Class: Car
Class: Truck
DataProperty: hasAirbags

Class: FamilyVehicle
    EquivalentTo: Car or Truck

Class: SafeFamilyVehicle
    EquivalentTo: FamilyVehicle that (hasAirbags value true)
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class .

:FamilyVehicle a owl:Class ;
    owl:equivalentClass [
        a owl:Class ;
        owl:unionOf ( :Car :Truck )
    ] .

:SafeFamilyVehicle a owl:Class ;
    owl:equivalentClass [
        a owl:Class ;
        owl:intersectionOf (
            :FamilyVehicle
            [
                a owl:Restriction ;
                owl:hasValue true ;
                owl:onProperty :hasAirbags
            ]
        )
    ] .

:Truck a owl:Class .

:hasAirbags a owl:DatatypeProperty .



### 5.3 `some`, `only` and `onlysome`

A restriction describes the individuals that have a certain kind of link through a property `p`. `p some C` is the individuals that have at least one `p` value in `C`. `p only C` is the individuals whose `p` values are all in `C`. It does not say there is any value, and a value that is stated is concluded to be in `C` and not rejected. `p onlysome C` is shorthand for `(p some C) and (p only C)` together.

Restrictions are normally used after `SubClassOf:` (every such individual has this property) or `EquivalentTo:` (an individual is in the class exactly when it has it).

Below, one restriction of each kind.

In [ ]:
_ = show("""
Class: Car
Class: Person
ObjectProperty: hasPassenger

Class: FullCar
    SubClassOf: Car
    SubClassOf: hasPassenger some Person

Class: PassengerOnlyCar
    SubClassOf: Car
    SubClassOf: hasPassenger only Person

Class: BalancedCar
    SubClassOf: Car
    SubClassOf: hasPassenger onlysome Person
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:BalancedCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Class ;
        owl:intersectionOf (
            [
                a owl:Restriction ;
                owl:onProperty :hasPassenger ;
                owl:someValuesFrom :Person
            ]
            [
                a owl:Restriction ;
                owl:allValuesFrom :Person ;
                owl:onProperty :hasPassenger
            ]
        )
    ] .

:Car a owl:Class .

:FullCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Restriction ;
        owl:onProperty :hasPassenger ;
        owl:someValuesFrom :Person
    ] .

:PassengerOnlyCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Restriction ;
        owl:allValuesFrom :Person ;
        owl:onProperty :hasPassenger
    ] .

:Person a owl:Class .

:hasPassenger a owl:ObjectProperty .



### 5.4 `value` and `Self`

`p value i` is the individuals linked by `p` to the specific individual `i`. `p Self` is the individuals that are linked by `p` to themselves.

Below, `AlicesCar` is a car owned by `Alice`, and `SelfMonitoringCar` is a car that monitors itself.

In [ ]:
_ = show("""
Class: Car
ObjectProperty: hasOwner
ObjectProperty: monitorsItself
Individual: Alice

Class: AlicesCar
    SubClassOf: Car
    SubClassOf: hasOwner value Alice

Class: SelfMonitoringCar
    SubClassOf: Car
    SubClassOf: monitorsItself Self
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Alice a owl:NamedIndividual .

:AlicesCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Restriction ;
        owl:hasValue :Alice ;
        owl:onProperty :hasOwner
    ] .

:Car a owl:Class .

:SelfMonitoringCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Restriction ;
        owl:hasSelf true ;
        owl:onProperty :monitorsItself
    ] .

:hasOwner a owl:ObjectProperty .

:monitorsItself a owl:ObjectProperty .



### 5.5 `min`, `max` and `exactly`

`p min n C`, `p max n C` and `p exactly n C` are the individuals with at least, at most, or exactly `n` distinct `p` values in `C`. The class is optional: with no class, any value counts. Compare `SmallCar` (qualified by a class) with `BigCar` (a count of any values).

Below, one example of each count.

In [ ]:
_ = show("""
Class: Car
Class: Person
ObjectProperty: hasPassenger
DataProperty: hasDoors

Class: Sedan
    SubClassOf: Car
    SubClassOf: hasDoors exactly 4

Class: SmallCar
    SubClassOf: Car
    SubClassOf: hasPassenger max 2 Person

Class: BigCar
    SubClassOf: Car
    SubClassOf: hasPassenger min 5
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:BigCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Restriction ;
        owl:minCardinality 5 ;
        owl:onProperty :hasPassenger
    ] .

:Car a owl:Class .

:Person a owl:Class .

:Sedan a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Restriction ;
        owl:cardinality 4 ;
        owl:onProperty :hasDoors
    ] .

:SmallCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Restriction ;
        owl:maxQualifiedCardinality 2 ;
        owl:onClass :Person ;
        owl:onProperty :hasPassenger
    ] .

:hasDoors a owl:DatatypeProperty .

:hasPassenger a owl:ObjectProperty .



### 5.6 Enumerations: `{a, b, c}`

An enumeration is a class defined by listing its members (`owl:oneOf`). The class has exactly those individuals and no others. `PrimaryColor EquivalentTo: {Red, Green, Blue}` means a color is a `PrimaryColor` if and only if it is `Red`, `Green` or `Blue`.

Below, an enumeration of three individuals.

In [ ]:
_ = show("""
Individual: Red
Individual: Green
Individual: Blue

Class: PrimaryColor
    EquivalentTo: {Red, Green, Blue}
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Blue a owl:NamedIndividual .

:Green a owl:NamedIndividual .

:PrimaryColor a owl:Class ;
    owl:equivalentClass [
        a owl:Class ;
        owl:oneOf ( :Red :Green :Blue )
    ] .

:Red a owl:NamedIndividual .



## 6. Data ranges

A data range is the set of data values a data property can have. It is the data-value counterpart of a class expression. A value outside the range of its property makes the ontology inconsistent for a DL reasoner.

### 6.1 Facets

`xsd:integer[>= 0, <= 400]` is a datatype restricted by facets: the integers from 0 to 400.

Below, a data property whose range is a restricted integer.

In [ ]:
_ = show("""
DataProperty: hasSpeedKmh
    Range: xsd:integer[>= 0, <= 400]
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:hasSpeedKmh a owl:DatatypeProperty ;
    rdfs:range [
        a rdfs:Datatype ;
        owl:onDatatype xsd:integer ;
        owl:withRestrictions (
            [
                xsd:minInclusive 0
            ]
            [
                xsd:maxInclusive 400
            ]
        )
    ] .



### 6.2 Unions and enumerations of values

`xsd:string or xsd:integer` is the union of two datatypes. `{"active", "retired"}` is an explicit list of the allowed values (`owl:oneOf` for data).

Below, one data property with each.

In [ ]:
_ = show("""
DataProperty: hasFuelType
    Range: xsd:string or xsd:integer

DataProperty: hasStatusCode
    Range: {"active", "retired"}
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:hasFuelType a owl:DatatypeProperty ;
    rdfs:range [
        a rdfs:Datatype ;
        owl:unionOf ( xsd:string xsd:integer )
    ] .

:hasStatusCode a owl:DatatypeProperty ;
    rdfs:range [
        a rdfs:Datatype ;
        owl:oneOf ( "active" "retired" )
    ] .



### 6.3 `Datatype:` frames

A `Datatype:` frame gives a name to a data range: `PositiveInt EquivalentTo: xsd:integer[>= 0]`. The name can then be used as a range.

Below, a named datatype.

In [ ]:
_ = show("""
Datatype: PositiveInt
    EquivalentTo: xsd:integer[>= 0]
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:PositiveInt a rdfs:Datatype ;
    owl:equivalentClass [
        a rdfs:Datatype ;
        owl:onDatatype xsd:integer ;
        owl:withRestrictions (
            [
                xsd:minInclusive 0
            ]
        )
    ] .



## 7. Annotations

Annotations are notes for people and tools. A reasoner ignores them: they do not change what the ontology means.

### 7.1 Frame-level `Annotations:`

An `Annotations:` clause on a frame is about the frame's own entity. It maps straight to plain triples with `rdfs:label`, `rdfs:comment`, or any other annotation property.

Below, a class is given a label and a comment.

In [ ]:
_ = show("""
Class: Car
    Annotations: rdfs:label "Car", rdfs:comment "A wheeled motor vehicle"
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class ;
    rdfs:comment "A wheeled motor vehicle" ;
    rdfs:label "Car" .



### 7.2 Inline per-axiom `Annotations:`

A more targeted form annotates one axiom and not the whole frame. An `Annotations:` block written inside another clause's list, immediately before one item, applies to just that axiom. The tree records this with an `owl:Axiom` node (`owl:annotatedSource`, `owl:annotatedProperty` and `owl:annotatedTarget`, plus the annotation itself), alongside the plain axiom triple and never instead of it. This is how to say who stated an axiom, or why.

Below, `Cat SubClassOf: Animal` carries a comment. `Pet` is also asserted as a superclass of `Cat`, but it is not annotated, because the annotation applies only to the item it precedes.

In [ ]:
_ = show("""
Class: Animal
Class: Pet
Class: Cat
    SubClassOf: Annotations: rdfs:comment "domesticated" Animal, Pet
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Animal a owl:Class .

:Cat a owl:Class ;
    rdfs:subClassOf :Animal, :Pet .

:Pet a owl:Class .

_:b0 a owl:Axiom ;
    rdfs:comment "domesticated" ;
    owl:annotatedProperty rdfs:subClassOf ;
    owl:annotatedSource :Cat ;
    owl:annotatedTarget :Animal .



## 8. Top-level Misc axioms

Some axioms relate several entities at once and do not belong to one frame. Each accepts two or more names, not just a pair.

### 8.1 Classes

`EquivalentClasses: A, B, C` means the classes have exactly the same members. `DisjointClasses: A, B, C` means no individual is in two of them.

Below, three equivalent classes, and three disjoint ones.

In [ ]:
_ = show("""
Class: Auto
Class: Car
Class: Motorcar
EquivalentClasses: Auto, Car, Motorcar

Class: Motorcycle
Class: Bicycle
DisjointClasses: Car, Motorcycle, Bicycle
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Auto a owl:Class ;
    owl:equivalentClass :Car .

:Bicycle a owl:Class .

:Car a owl:Class ;
    owl:equivalentClass :Motorcar .

:Motorcar a owl:Class .

:Motorcycle a owl:Class .

_:b0 a owl:AllDisjointClasses ;
    owl:members ( :Car :Motorcycle :Bicycle ) .



### 8.2 Properties

`EquivalentProperties: p, q` means the properties link the same pairs of individuals. `DisjointProperties: p, q` means no two individuals are linked by both.

Below, two equivalent properties, and two disjoint ones.

In [ ]:
_ = show("""
ObjectProperty: hasOwner
ObjectProperty: ownedBy
EquivalentProperties: hasOwner, ownedBy

ObjectProperty: hasPart
DisjointProperties: hasOwner, hasPart
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:hasOwner a owl:ObjectProperty ;
    owl:equivalentProperty :ownedBy ;
    owl:propertyDisjointWith :hasPart .

:hasPart a owl:ObjectProperty .

:ownedBy a owl:ObjectProperty .



### 8.3 Individuals

`SameIndividual: a, b` means the names refer to the same individual. `DifferentIndividuals: a, b` means they are different individuals.

Below, two names for one car, and a different car.

In [ ]:
_ = show("""
Individual: mycar
Individual: myauto
SameIndividual: mycar, myauto

Individual: yourcar
DifferentIndividuals: mycar, yourcar
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:myauto a owl:NamedIndividual .

:mycar a owl:NamedIndividual ;
    owl:differentFrom :yourcar ;
    owl:sameAs :myauto .

:yourcar a owl:NamedIndividual .

